## proceso para optener la calificacion camel 
* se optienen los pesos
* se optiene la calificacion con los rangos de los cuartiles
* se hace el modelo camel
* se optiene la calificacion 

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
import pandas as pd

## PCA

In [3]:
def pesos_pca_grupo_coops(
    df,
    lista_coops,
    col_nombre="ID_cooperativa",
    n_componentes=3
):
    
    # filtrar cooperativas
    df_filtrado = df[df[col_nombre].isin(lista_coops)].copy()
    
    if df_filtrado.empty:
        raise ValueError("No hay datos para las cooperativas indicadas")
    
    # LIMPIEZA 
    df_filtrado["valor"] = (
        df_filtrado["valor"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    
    df_filtrado["valor"] = pd.to_numeric(df_filtrado["valor"], errors="coerce")
    
    # pivot (matriz indicadores)
    df_pivot = df_filtrado.pivot_table(
        index=col_nombre,
        columns="ID_indicador",
        values="valor",
        aggfunc="mean"
    )
    
    # imputar faltantes
    imputer = SimpleImputer(strategy="mean")
    X_imputed = imputer.fit_transform(df_pivot)
    
    # escalar
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imputed)
    
    # PCA
    pca = PCA(n_components=min(n_componentes, df_pivot.shape[1]))
    pca.fit(X_scaled)
    
    loadings = pd.DataFrame(
        pca.components_.T,
        index=df_pivot.columns
    )
    
    pesos = (loadings**2).mean(axis=1)
    
    # porcentaje
    pesos = (pesos / pesos.sum()) * 100
    
    return pesos.sort_values(ascending=False)

In [31]:
datos=pd.read_excel("../tablas/registros_solvencia_actualizado.xlsx")
datos.drop(columns=["Unnamed: 0"], inplace=True)
datos=datos.loc[datos["ano"]>=2020]
datos.head()

,ID_registro,ID_indicador,ID_cooperativa,ano,mes,valor
25344,25345,Quebranto Patrimonial,CAJA COOPERATIVA CREDICOOP,2020,1,0.0
25345,25346,Quebranto Patrimonial,CAJA COOPERATIVA PETROLERA,2020,1,0.0
25346,25347,Quebranto Patrimonial,CASA NACIONAL DEL PROFESOR,2020,1,0.0
25347,25348,Quebranto Patrimonial,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,2020,1,0.0
25348,25349,Quebranto Patrimonial,COOPERATIVA AHORRO Y CREDITO GOMEZ PLATA LTDA.,2020,1,0.0


In [11]:
names_coop = datos["ID_cooperativa"].tolist()
pesos_pca_grupo_coops(df=datos, lista_coops=names_coop)

ID_indicador
Quebranto Patrimonial                                                                     1.202691e+01
Indicador de Margen Financiero de Operación                                               1.119254e+01
Indicador de rentabilidad sobre recursos propios - ROE                                    1.027229e+01
Estructura de Balance                                                                     9.915166e+00
Indicador de Cobertura de la Cartera Total en Riesgo                                      7.855362e+00
Relación entre el Capital Institucional y el Activo Total                                 7.551783e+00
Indicador de rentabilidad sobre el capital invertido - ROIC                               6.721697e+00
Indicador de calidad por riesgo                                                           6.350810e+00
Indicador de margen neto                                                                  6.263051e+00
Relación entre Aportes sociales mínimos no reducibles y Capi

## Calificacion 

In [ ]:
def tabla_percentiles_globales(df: pd.DataFrame):
    if df.empty:
        return pd.DataFrame()

    df = df.copy()

    df['percentil'] = (
        df
        .groupby('ID_indicador')['valor']
        .rank(pct=True) * 100
    )

    df['decil'] = (df['percentil'] // 10).astype(int) + 1
    df['decil'] = df['decil'].clip(upper=10)  

    tabla = (
        df
        .groupby(['ID_indicador', 'decil'])['valor']
        .max()
        .unstack(fill_value=None)
    )

    tabla.columns = [f"P{int(col*10)}" for col in tabla.columns]

    return tabla

In [28]:
tabla_percentiles_globales(datos)

,P10,P20,P30,P40,P50,P60,P70,P80,P90,P100
ID_indicador,,,,,,,,,,
Activo Productivo,0.000000,0.721226,0.782721,0.816566,0.843184,0.864405,0.883183,0.905425,0.936185,1.036441
Estructura de Balance,0.000000,1.107473,1.222317,1.329067,1.427431,1.551734,1.783026,2.230817,3.011196,10.765512
Indicador de Cobertura de la Cartera Total en Riesgo,0.000000,0.024611,0.030502,0.037168,0.044326,0.054433,0.066287,0.077870,0.098580,0.235701
Indicador de Cobertura individual de la cartera improductiva para la cartera en Riesgo,0.000000,0.255854,0.376413,0.461996,0.533330,0.595869,0.663457,0.732604,0.821393,2.702713
Indicador de Margen Financiero de Operación,0.000000,0.533184,0.625496,0.679795,0.724092,0.761047,0.803083,0.851783,0.897019,0.997984
Indicador de Margen Operacional,-0.079885,-0.003161,0.000000,0.025453,0.066143,0.107839,0.146674,0.192808,0.256390,0.710488
Indicador de Riesgo de Liquidez - IRL,NaN,0.000000,187.550000,201.570000,213.380000,223.040000,226.450000,252.030000,261.180000,308.910000
Indicador de calidad por riesgo,0.000000,0.026531,0.038659,0.049625,0.058684,0.070069,0.084292,0.102265,0.138733,1.016518
Indicador de calidad por riesgo con castigos,0.000000,0.000862,0.004374,0.011388,0.020182,0.029728,0.043201,0.058114,0.088265,0.314842


In [32]:
def calcular_calificaciones_formato_lista(df: pd.DataFrame):
    if df.empty:
        return []

    columnas_necesarias = {
        'ID_indicador', 'ID_cooperativa', 'valor'
    }
    if not columnas_necesarias.issubset(df.columns):
        raise ValueError("Faltan columnas necesarias")

    def asignar_calificacion(percentil):
        if percentil <= 10: return 1
        elif percentil <= 20: return 2
        elif percentil <= 30: return 3
        elif percentil <= 40: return 4
        elif percentil <= 50: return 5
        elif percentil <= 60: return 6
        elif percentil <= 70: return 7
        elif percentil <= 80: return 8
        elif percentil <= 90: return 9
        else: return 10

    df = df.copy()

    resultados_temp = []

    for indicador in df['ID_indicador'].unique():
        df_ind = df[df['ID_indicador'] == indicador].copy()

        df_ind['percentil'] = df_ind['valor'].rank(pct=True) * 100
        df_ind['calificacion'] = df_ind['percentil'].apply(asignar_calificacion)

        promedio = (
            df_ind
            .groupby('ID_cooperativa')['calificacion']
            .mean()
            .round()
            .astype(int)
        )

        for coop, calif in promedio.items():
            resultados_temp.append({
                "ID_cooperativa": str(coop),
                "ID_indicador": str(indicador),
                "calificacion": int(calif)
            })

    resultado_final = {}

    for row in resultados_temp:
        coop = row["ID_cooperativa"]
        ind = row["ID_indicador"]
        cal = row["calificacion"]

        if coop not in resultado_final:
            resultado_final[coop] = {
                "ID_cooperativa": coop,
                "calificaciones": {}
            }

        resultado_final[coop]["calificaciones"][ind] = cal

    return list(resultado_final.values())

In [30]:
calificaciones = calcular_calificaciones_formato_lista(datos)
calificaciones

[{'ID_cooperativa': 'CAJA COOPERATIVA CREDICOOP',
  'calificaciones': {'Quebranto Patrimonial': 5,
   'Relación Solvencia': 5,
   'Relación entre Aportes sociales mínimos no reducibles y Capital Social': 8,
   'Relación entre el Capital Institucional y el Activo Total': 6,
   'Indicador de Cobertura de la Cartera Total en Riesgo': 6,
   'Indicador de calidad por riesgo': 7,
   'Indicador de calidad por riesgo con castigos': 9,
   'Activo Productivo': 5,
   'Indicador de Cobertura individual de la cartera improductiva para la cartera en Riesgo': 6,
   'Estructura de Balance': 7,
   'Indicador de Margen Financiero de Operación': 6,
   'Indicador de Margen Operacional': 4,
   'Indicador de relación entre las obligaciones financieras y el pasivo total': 6,
   'Indicador de margen neto': 2,
   'Indicador de rentabilidad sobre el capital invertido - ROIC': 2,
   'Indicador de rentabilidad sobre recursos propios - ROE': 2,
   'Indicador de Riesgo de Liquidez - IRL': 6}},
 {'ID_cooperativa': '